In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("records").getOrCreate()


25/03/17 07:05:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
df = spark.sql("SELECT count(*) FROM demo.nyc.taxis_1M_50COLUMNS_DELETE")
df.show()

+--------+
|count(1)|
+--------+
|  999000|
+--------+



In [5]:
from pyspark.sql import SparkSession
import time

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Delete N Rows from Iceberg Table") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.memoryOverhead", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

# Define the number of rows to delete
N = 1000  # Change this value as needed

# Measure start time
start_time = time.time()

# Use DELETE statement with a valid column name
spark.sql(f"""
    DELETE FROM demo.nyc.taxis_1M_50COLUMNS_DELETE 
    WHERE extra_col_0 IN (
        SELECT extra_col_0 FROM demo.nyc.taxis_1M_50COLUMNS_DELETE LIMIT {N}
    )
""")

time_taken = time.time() - start_time
print(f"{N} rows have been deleted successfully in {time_taken:.2f} seconds.")


1000 rows have been deleted successfully in 39.30 seconds.


In [24]:
from pyspark.sql import SparkSession
import time

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Delete Column from Iceberg Table") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.memoryOverhead", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

# Measure start time
start_time = time.time()
spark.sql("""
    ALTER TABLE demo.nyc.taxis_1M_50COLUMNS_DELETE
    DROP COLUMN extra_col_4
""")
time_taken = time.time() - start_time
print(f"Column extra_col_4 has been deleted successfully in {time_taken:.2f} seconds.")


Column extra_col_4 has been deleted successfully in 0.17 seconds.


In [30]:
from pyspark.sql import SparkSession
import time

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Delete Values in Column from Iceberg Table") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.memoryOverhead", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

# Define condition for deletion (e.g., delete values where column equals some value)
condition = "extra_col_1 < 100" 
start_time = time.time()
spark.sql(f"""
    UPDATE demo.nyc.taxis_10000_50COLUMNS_DELETE
    SET extra_col_1 = NULL
    WHERE {condition}
""")
time_taken = time.time() - start_time
print(f"Values in column extra_col_1 have been deleted (set to NULL) in {time_taken:.2f} seconds.")


Values in column extra_col_1 have been deleted (set to NULL) in 1.48 seconds.


In [5]:
df = spark.sql("SELECT count(*) FROM demo.nyc.taxis_1K_product_part2")
df.show()

+--------+
|count(1)|
+--------+
|    1000|
+--------+



In [6]:

df = spark.sql("SELECT * FROM demo.nyc.taxis_1K_product_part2.files")
df.show()

25/03/10 07:23:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+-----------+-------+---------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|content|           file_path|file_format|spec_id|partition|record_count|file_size_in_bytes|        column_sizes|        value_counts|   null_value_counts|nan_value_counts|        lower_bounds|        upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|    readable_metrics|
+-------+--------------------+-----------+-------+---------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|      0|s3://warehouse/ny...|    PARQUET|      0|     {CV}|         350|             61782|{1 -> 2354, 2 -> ...|{1 -> 350, 2 -> 3

In [9]:

df = spark.sql("SELECT * FROM demo.nyc.taxis_1K_product_p2_new.files")
df.show()

+-------+--------------------+-----------+-------+---------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|content|           file_path|file_format|spec_id|partition|record_count|file_size_in_bytes|        column_sizes|        value_counts|   null_value_counts|nan_value_counts|        lower_bounds|        upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|    readable_metrics|
+-------+--------------------+-----------+-------+---------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|      0|s3://warehouse/ny...|    PARQUET|      0|     {CV}|         331|             59451|{1 -> 2237, 2 -> ...|{1 -> 331, 2 -> 3

In [10]:


df = spark.sql("SELECT * FROM demo.nyc.taxis_1K_product_p2_new.partitions")
df.show()

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+--------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|     last_updated_at|last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+--------------------+------------------------+
|     {CV}|      0|         331|         1|                        59451|                           0|                         0|                           0|                         0|2025-03-07 09:37:...|     2455502788856819042|
|     {TW}|      0|         341|         1|                        60696

In [12]:

df = spark.sql("SELECT column_name, partition_transform FROM demo.nyc.taxis_1K_product_p2_new.partitions")
df.show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `column_name` cannot be resolved. Did you mean one of the following? [`spec_id`, `file_count`, `partition`, `record_count`, `last_updated_at`].; line 1 pos 7;
'Project ['column_name, 'partition_transform]
+- SubqueryAlias demo.nyc.taxis_1K_product_p2_new.partitions
   +- RelationV2[partition#731, spec_id#732, record_count#733L, file_count#734, total_data_file_size_in_bytes#735L, position_delete_record_count#736L, position_delete_file_count#737, equality_delete_record_count#738L, equality_delete_file_count#739, last_updated_at#740, last_updated_snapshot_id#741L] demo.nyc.taxis_1K_product_p2_new.partitions demo.nyc.taxis_1K_product_p2_new.partitions


In [13]:
df = spark.sql("DESCRIBE FORMATTED demo.nyc.taxis_1K_product_p2_new")
df.show(truncate=False)


+------------+---------+-------+
|col_name    |data_type|comment|
+------------+---------+-------+
|extra_col_0 |string   |NULL   |
|extra_col_1 |int      |NULL   |
|extra_col_2 |string   |NULL   |
|extra_col_3 |date     |NULL   |
|extra_col_4 |string   |NULL   |
|extra_col_5 |int      |NULL   |
|extra_col_6 |string   |NULL   |
|extra_col_7 |date     |NULL   |
|extra_col_8 |string   |NULL   |
|extra_col_9 |int      |NULL   |
|extra_col_10|string   |NULL   |
|extra_col_11|date     |NULL   |
|extra_col_12|string   |NULL   |
|extra_col_13|int      |NULL   |
|extra_col_14|string   |NULL   |
|extra_col_15|date     |NULL   |
|extra_col_16|string   |NULL   |
|extra_col_17|int      |NULL   |
|extra_col_18|string   |NULL   |
|extra_col_19|date     |NULL   |
+------------+---------+-------+
only showing top 20 rows



In [14]:
df = spark.sql("SELECT DISTINCT partition FROM demo.nyc.taxis_1K_product_p2_new.files")
df.show(truncate=False)

+---------+
|partition|
+---------+
|{CAR}    |
|{CV}     |
|{TW}     |
+---------+

